# N-1 Contingency Analysis Notebook
This notebook demonstrates running N-1 analysis on the power system

In [123]:
import os
import sys
from pathlib import Path

# Add src to path for imports
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

# Verify we're in the right directory
print(f"Current working directory: {os.getcwd()}")
print(f"Repository root: {REPO_ROOT}")

Current working directory: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky\example
Repository root: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky


In [124]:
from power_system_simulation.lv_grid_analytics import LVGridAnalytics
from power_system_simulation.N_minus_1 import InvalidLineOutageError

print("✓ Imports successful")

✓ Imports successful


In [125]:
# Define test data paths
TEST_DATA_DIR = REPO_ROOT / "tests" / "big_network" / "big_network" / "input"

print(f"Test data directory: {TEST_DATA_DIR}")
print(f"Directory exists: {TEST_DATA_DIR.exists()}")

# List files in the directory
if TEST_DATA_DIR.exists():
    print("\nFiles in test directory:")
    for file in sorted(TEST_DATA_DIR.iterdir()):
        print(f"  - {file.name}")

Test data directory: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky\tests\big_network\big_network\input
Directory exists: True

Files in test directory:
  - active_power_profile.parquet
  - ev_active_power_profile.parquet
  - input_network_data.json
  - meta_data.json
  - reactive_power_profile.parquet


In [126]:
# Create LVGridAnalytics instance
analytics = LVGridAnalytics(
    grid_path=str(TEST_DATA_DIR / "input_network_data.json"),
    meta_data = str(TEST_DATA_DIR / "meta_data.json"),
    active_load_profile_path=str(TEST_DATA_DIR / "active_power_profile.parquet"),
    reactive_load_profile_path=str(TEST_DATA_DIR / "reactive_power_profile.parquet"),
    ev_profile_path=str(TEST_DATA_DIR / "ev_active_power_profile.parquet"),
)

print("✓ LVGridAnalytics instance created")

✓ LVGridAnalytics instance created


In [127]:
# Validate inputs
analytics.validate_inputs()
print("✓ Inputs validated")

✓ Inputs validated


In [128]:
# Check the grid structure
print("Dataset info:")
print(f"  Lines: {len(analytics._dataset['line'])} lines")
print(f"  Nodes: {len(analytics._dataset['node'])} nodes")
print(f"  Loads: {len(analytics._dataset['sym_load'])} loads")
print(f"\nActive load profile shape: {analytics._active_load_profiles.shape}")
print(f"Reactive load profile shape: {analytics._reactive_load_profiles.shape}")

Dataset info:
  Lines: 807 lines
  Nodes: 802 nodes
  Loads: 400 loads

Active load profile shape: (35040, 400)
Reactive load profile shape: (35040, 400)


In [129]:
# Run N-1 analysis for line 16
print("Running N-1 contingency analysis for line 16...\n")
results_line_16 = analytics.n_minus_one(outage_line_id=1204)

print(f"Results shape: {results_line_16.shape}")
print(f"\nColumns: {list(results_line_16.columns)}")
print(f"\nResults:\n{results_line_16}")

Running N-1 contingency analysis for line 16...



KeyboardInterrupt: 

In [ ]:
# Run N-1 analysis for line 20
print("Running N-1 contingency analysis for line 20...\n")
results_line_18 = analytics.n_minus_one(outage_line_id=20)

print(f"Results shape: {results_line_18.shape}")
print(f"\nResults:\n{results_line_18}")

Running N-1 contingency analysis for line 20...

Results shape: (1, 4)

Results:
   Alternative_Line_ID  Max_Loading  Max_Loading_Line_ID Max_Loading_Timestamp
0                   24     0.001662                   21   2025-01-07 10:30:00


In [ ]:
# Test with invalid line ID
print("Testing with invalid line ID (999)...\n")
try:
    results_invalid = analytics.n_minus_one(outage_line_id=999)
except InvalidLineOutageError as e:
    print(f"✓ Correctly caught error: {e}")

Testing with invalid line ID (999)...

✓ Correctly caught error: Line ID 999 not found in the dataset.


In [ ]:
# Summary
print("\n" + "=" * 60)
print("N-1 CONTINGENCY ANALYSIS SUMMARY")
print("=" * 60)
print("\nLine 16 Outage:")
print(f"  - Number of alternatives: {len(results_line_16)}")
if len(results_line_16) > 0:
    print(
        f"  - Max loading range: "
        f"{results_line_16['Max_Loading'].min():.4f} - "
        f"{results_line_16['Max_Loading'].max():.4f}"
    )
    print(f"  - Alternative lines: {sorted(results_line_16['Alternative_Line_ID'].tolist())}")

print("\nLine 20 Outage:")
print(f"  - Number of alternatives: {len(results_line_18)}")
if len(results_line_18) > 0:
    print(
        f"  - Max loading range: "
        f"{results_line_18['Max_Loading'].min():.4f} - "
        f"{results_line_18['Max_Loading'].max():.4f}"
    )
    print(f"  - Alternative lines: {sorted(results_line_18['Alternative_Line_ID'].tolist())}")

print("\n✓ N-1 analysis completed successfully!")


N-1 CONTINGENCY ANALYSIS SUMMARY

Line 16 Outage:
  - Number of alternatives: 1
  - Max loading range: 0.0017 - 0.0017
  - Alternative lines: [24]

Line 20 Outage:
  - Number of alternatives: 0

✓ N-1 analysis completed successfully!
